# Hands-on OHDSI Athena enrichment with suMEDS

This notebook enriches individual MEDS codes, demonstrates configurable ancestor, descendant, and sibling codes, uses existing MEDS parent references as a fallback, enriches the MIMIC-IV demo metadata, and optionally runs a complete privacy-aware summary. No source files are modified.

## Start Jupyter

From the repository root, choose one source. For the included Docker PostgreSQL service:

```bash
# docker compose up -d db
set -a; source .env; set +a
export PGPASSWORD="$POSTGRES_PASSWORD"
uv pip install jupyter  # local development environment only
uv run jupyter lab examples/athena_enrichment_demo.ipynb
```

For local Athena files, set `ATHENA_CSV=/absolute/path/to/athena` before starting Jupyter. The directory needs tab-delimited `CONCEPT.csv` and `CONCEPT_ANCESTOR.csv`. The notebook otherwise defaults to PostgreSQL at `127.0.0.1:5432/omop`.

In [ ]:
from pathlib import Path
import os

import polars as pl

from sumeds import (
    EnrichmentConfig,
    SummaryConfig,
    enrich_file,
    enrich_metadata,
    summarize,
)

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

athena_csv = os.getenv("ATHENA_CSV")
if "PGPASSWORD" not in os.environ and "POSTGRES_PASSWORD" in os.environ:
    os.environ["PGPASSWORD"] = os.environ["POSTGRES_PASSWORD"]
athena_source = (
    {"csv_dir": athena_csv}
    if athena_csv
    else {
        "postgres": os.getenv(
            "ATHENA_POSTGRES",
            "postgresql://postgres@127.0.0.1:5432/omop",
        )
    }
)
athena = EnrichmentConfig(**athena_source)
hierarchy_demo = EnrichmentConfig(
    **athena_source,
    child_codes=True,
    sibling_codes=True,
    child_depth=3,
)
print("Using local Athena CSV files" if athena_csv else "Using Athena PostgreSQL")

## 1. Enrich a few codes

The first two rows use `VOCABULARY//CODE//marker`. The third uses `VOCABULARY//VERSION//CODE`. The MIMIC-style lab code is local, so its existing `LOINC/5803-2` parent reference supplies the Athena lookup key. Existing descriptions and relationship arrays are retained and merged without duplicates. Parent expansion is enabled by default and follows ancestors through the root; set `parent_codes=False` to disable it. This small example also enables descendants through `child_depth=3` and siblings (direct children of applicable ancestors).

In [ ]:
codes = pl.DataFrame(
    {
        "code": [
            "CMS Place of Service//02//end",
            "CMS Place of Service//02//start",
            "ICD10CM//2024//I95.1",
            "LAB//51491//units",
            "NOT_A_VOCABULARY//missing",
        ],
        "description": [None, None, None, "Existing lab description", None],
        "parent_codes": [None, None, None, ["LOINC//5803-2"], None],
    },
    schema_overrides={"parent_codes": pl.List(pl.String)},
)

enriched_codes = enrich_metadata(codes.lazy(), hierarchy_demo).collect()
enriched_codes.select(
    "code",
    "concept_id",
    "concept_code",
    "description",
    "domain_id",
    "standard_concept",
    "parent_codes",
    "child_codes",
    "sibling_codes",
)

In [ ]:
enriched_codes.with_columns(
    pl.col("concept_id").is_not_null().alias("athena_match")
).group_by("athena_match").len()

## 2. Enrich the included MIMIC-IV demo metadata

MIMIC event codes are dataset-local, but most rows already link to an Athena concept through `parent_codes`. This operation reads metadata only—no patient event shards. It uses `athena`, which keeps the default all-ancestor enrichment without the opt-in child and sibling expansion used above.

In [ ]:
mimic_root = repo_root / "tests/resources/MIMICIV_DEMO/MEDS_cohort"
mimic_codes_path = mimic_root / "metadata/codes.parquet"
if not mimic_codes_path.exists():
    raise FileNotFoundError(
        f"Demo metadata not found at {mimic_codes_path}; run from the repository root."
    )

mimic_enriched = enrich_metadata(pl.scan_parquet(mimic_codes_path), athena).collect()
match_count = mimic_enriched["concept_id"].is_not_null().sum()
print(f"Matched {match_count:,} of {mimic_enriched.height:,} metadata rows")
mimic_enriched.filter(pl.col("concept_id").is_not_null()).select(
    "code", "concept_id", "concept_code", "domain_id", "standard_concept"
).head(10)

## 3. Write an enriched metadata table atomically

`enrich_file` supports Parquet, CSV, JSON, JSONL, and NDJSON. It requires a separate output path and leaves the source untouched.

In [ ]:
demo_output = repo_root / "demo-output/mimic-codes-enriched.parquet"
demo_output.parent.mkdir(exist_ok=True)
enrich_file(mimic_codes_path, demo_output, athena)
pl.scan_parquet(demo_output).select(
    pl.len().alias("rows"),
    pl.col("concept_id").is_not_null().sum().alias("athena_matches"),
).collect()

In [ ]:
pl.scan_parquet(demo_output).collect()

## 4. Optional full summary

Set `RUN_FULL_SUMMARY = True` to scan MIMIC events, apply the privacy policy, and enrich only released rows. Rare source codes are masked before Athena lookup.

In [ ]:
RUN_FULL_SUMMARY = False

if RUN_FULL_SUMMARY:
    summary_path = repo_root / "demo-output/mimic-summary-enriched.parquet"
    summarize(
        mimic_root,
        summary_path,
        SummaryConfig(min_subjects=20, enrichment=athena),
    )
    display(pl.read_parquet(summary_path).head(10))
else:
    print("Skipped. Set RUN_FULL_SUMMARY = True to run it.")

## Cleanup

Uncomment the next cell to remove files created by this notebook.

In [ ]:
# import shutil
# shutil.rmtree(repo_root / "demo-output", ignore_errors=True)